

This notebook converts an inferred `.trees` or `.trees.tsz` file into a synthetic full ARG, builds the compact stepwise trace, advances one mutable cursor to the terminal state, and keeps a separate cursor for traceback. Synthetic recombination times are midpoint imputations; the marginal genealogies come from the input tree sequence.

In [1]:
from __future__ import annotations

import gc
import os
from pathlib import Path
import resource
import sys
import threading
import time

from IPython.display import display
import numpy as np
import tskit


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start).expanduser().resolve()
    candidates = (start, *start.parents)
    fallback = Path("/Users/pratik/Documents/work/aim3/simpliied")
    for path in (*candidates, fallback):
        if (path / "arg/new_rl/trace.py").is_file() and (
            path / "argscape/synthetic_full_arg.py"
        ).is_file():
            return path
    raise RuntimeError("Could not locate the workspace containing arg/new_rl and argscape")


WORKSPACE_ROOT = find_workspace_root()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

import arg.new_rl as new_rl_module
from arg.new_rl import FastARGState, build_fast_trace_from_full_arg
from argscape import (
    NODE_IS_RE_EVENT,
    SyntheticFullARGResult,
    build_synthetic_full_arg,
)

EXPECTED_NEW_RL = (WORKSPACE_ROOT / "arg/new_rl/__init__.py").resolve()
RESOLVED_NEW_RL = Path(new_rl_module.__file__).resolve()
if RESOLVED_NEW_RL != EXPECTED_NEW_RL:
    raise RuntimeError(f"Expected {EXPECTED_NEW_RL}, imported {RESOLVED_NEW_RL}")

print(f"Python: {sys.executable}", flush=True)
print(f"new_rl: {RESOLVED_NEW_RL}", flush=True)

Python: /opt/anaconda3/envs/phylogfn_311/bin/python
new_rl: /Users/pratik/Documents/work/aim3/simpliied/arg/new_rl/__init__.py


## Configuration

The defaults run the full chr2 workflow. Environment variables make it possible to execute the same notebook against a smaller fixture for validation.

In [2]:
def environment_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    return value.strip().lower() in {"1", "true", "yes", "on"}


DEFAULT_TREE_PATH = Path(
    "/Users/pratik/.lorax/projects/1000Genomes/1kg_chr2.trees.tsz"
)
TREE_PATH = Path(os.environ.get("ARG_TREES_PATH", DEFAULT_TREE_PATH)).expanduser()
SPLIT_RULE = os.environ.get("ARG_SPLIT_RULE", "balanced")
CURSOR_CHUNK_SIZE = int(os.environ.get("ARG_CURSOR_CHUNK_SIZE", "65536")) ## compiled code handles this many events in njit before returning to Python
PROGRESS_EVENTS = int(os.environ.get("ARG_PROGRESS_EVENTS", "1000000"))
HEARTBEAT_SECONDS = float(os.environ.get("ARG_HEARTBEAT_SECONDS", "30"))
TRACEBACK_EVENTS = int(os.environ.get("ARG_TRACEBACK_EVENTS", "10000"))
VERIFY_MARGINALS = environment_flag("ARG_VERIFY_MARGINALS", False)
SAVE_SYNTHETIC = environment_flag("ARG_SAVE_SYNTHETIC", False)

input_stem = TREE_PATH.name
for suffix in (".trees.tsz", ".trees", ".tsz"):
    if input_stem.endswith(suffix):
        input_stem = input_stem[: -len(suffix)]
        break
SYNTHETIC_OUTPUT_PATH = Path(
    os.environ.get(
        "ARG_SYNTHETIC_OUTPUT_PATH",
        TREE_PATH.parent / f"{input_stem}_synthetic_full_arg.trees",
    )
).expanduser()

if CURSOR_CHUNK_SIZE <= 0:
    raise ValueError("CURSOR_CHUNK_SIZE must be positive")
if PROGRESS_EVENTS < 0:
    raise ValueError("PROGRESS_EVENTS must be nonnegative")
if TRACEBACK_EVENTS < 0:
    raise ValueError("TRACEBACK_EVENTS must be nonnegative")
if SPLIT_RULE not in {"balanced", "left_to_right"}:
    raise ValueError("SPLIT_RULE must be 'balanced' or 'left_to_right'")
if not TREE_PATH.is_file():
    raise FileNotFoundError(TREE_PATH)

configuration = {
    "tree_path": str(TREE_PATH),
    "split_rule": SPLIT_RULE,
    "cursor_chunk_size": CURSOR_CHUNK_SIZE,
    "progress_events": PROGRESS_EVENTS,
    "heartbeat_seconds": HEARTBEAT_SECONDS,
    "traceback_events": TRACEBACK_EVENTS,
    "verify_marginals": VERIFY_MARGINALS,
    "save_synthetic": SAVE_SYNTHETIC,
    "synthetic_output_path": str(SYNTHETIC_OUTPUT_PATH),
}
configuration

{'tree_path': '/Users/pratik/.lorax/projects/1000Genomes/1kg_chr2.trees.tsz',
 'split_rule': 'balanced',
 'cursor_chunk_size': 65536,
 'progress_events': 1000000,
 'heartbeat_seconds': 30.0,
 'traceback_events': 10000,
 'verify_marginals': False,
 'save_synthetic': False,
 'synthetic_output_path': '/Users/pratik/.lorax/projects/1000Genomes/1kg_chr2_synthetic_full_arg.trees'}

In [3]:
def emit(message):
    print(message, flush=True)


def max_rss_gib():
    rss = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    if sys.platform == "darwin":
        return rss / 1024**3
    return rss / 1024**2


def run_stage(label, operation):
    started = time.perf_counter()
    stopped = threading.Event()
    succeeded = False

    def report_heartbeat():
        while not stopped.wait(HEARTBEAT_SECONDS):
            elapsed = time.perf_counter() - started
            emit(f"{label}: running | elapsed={elapsed:.1f}s | max_rss={max_rss_gib():.2f} GiB")

    emit(f"{label}: started | max_rss={max_rss_gib():.2f} GiB")
    reporter = None
    if HEARTBEAT_SECONDS > 0:
        reporter = threading.Thread(target=report_heartbeat, daemon=True)
        reporter.start()
    try:
        result = operation()
        succeeded = True
        return result
    finally:
        stopped.set()
        if reporter is not None:
            reporter.join()
        elapsed = time.perf_counter() - started
        status = "finished" if succeeded else "failed"
        emit(f"{label}: {status} | elapsed={elapsed:.1f}s | max_rss={max_rss_gib():.2f} GiB")


def load_tree_sequence(path):
    path = Path(path)
    if path.suffix == ".tsz":
        import tszip

        return tszip.decompress(path)
    return tskit.load(str(path))


def tree_sequence_summary(ts):
    return {
        "sequence_length": float(ts.sequence_length),
        "trees": int(ts.num_trees),
        "samples": int(ts.num_samples),
        "nodes": int(ts.num_nodes),
        "edges": int(ts.num_edges),
        "sites": int(ts.num_sites),
        "mutations": int(ts.num_mutations),
    }

def state_summary(state):
    return {
        "step": int(state.step),
        "num_steps": int(state.trace.num_steps),
        "current_time": float(state.current_time),
        "is_terminal": bool(state.is_terminal),
        "visible_nodes": int(state.visible_node_ids.size),
        "visible_edges": int(state.visible_edge_ids.size),
        "active_lineages": int(state.active_count),
        "active_segments": int(state.segment_count),
    }

## Load and Build the Synthetic Full ARG

If the input already contains explicit recombination-event nodes, it is retained unchanged so recombination nodes are not synthesized twice.

In [4]:
source_ts = run_stage("Load input tree sequence", lambda: load_tree_sequence(TREE_PATH))
input_summary = tree_sequence_summary(source_ts)
source_flags = np.asarray(source_ts.nodes_flags, dtype=np.uint32)
source_recombination_nodes = np.flatnonzero((source_flags & NODE_IS_RE_EVENT) != 0)
input_was_synthetic = bool(source_recombination_nodes.size)

if input_was_synthetic:
    if source_recombination_nodes.size % 2:
        raise ValueError("Existing synthetic ARG has an odd number of recombination nodes")
    emit("Explicit recombination nodes detected; synthetic conversion is not repeated.")
    synthetic_metadata = {
        "source": "existing_synthetic_full_arg",
        "time_rule": "existing",
        "split_rule": "existing",
        "original_num_nodes": int(source_ts.num_nodes),
        "original_num_edges": int(source_ts.num_edges),
        "original_num_trees": int(source_ts.num_trees),
        "synthetic_recombination_event_count": int(source_recombination_nodes.size // 2),
        "synthetic_recombination_node_count": int(source_recombination_nodes.size),
        "augmented_num_nodes": int(source_ts.num_nodes),
        "augmented_num_edges": int(source_ts.num_edges),
        "augmented_num_trees": int(source_ts.num_trees),
        "already_synthetic": True,
    }
    synthetic_result = SyntheticFullARGResult(
        tree_sequence=source_ts,
        candidates=(),
        events=(),
        metadata=synthetic_metadata,
    )
else:
    synthetic_result = run_stage(
        "Building synthetic full ARG",
        lambda: build_synthetic_full_arg(source_ts, split_rule=SPLIT_RULE),
    )

synthetic_arg = synthetic_result.tree_sequence
synthetic_summary = tree_sequence_summary(synthetic_arg)
synthetic_recombination_nodes = np.flatnonzero(
    (np.asarray(synthetic_arg.nodes_flags, dtype=np.uint32) & NODE_IS_RE_EVENT) != 0
)

assert synthetic_summary["trees"] == input_summary["trees"]
assert synthetic_summary["samples"] == input_summary["samples"]
assert synthetic_recombination_nodes.size == synthetic_result.metadata[
    "synthetic_recombination_node_count"
]
assert synthetic_recombination_nodes.size % 2 == 0

if SAVE_SYNTHETIC:
    SYNTHETIC_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    run_stage(
        "Save synthetic full ARG",
        lambda: synthetic_arg.dump(str(SYNTHETIC_OUTPUT_PATH)),
    )

validation_source_ts = source_ts if VERIFY_MARGINALS else None
if not VERIFY_MARGINALS and synthetic_arg is not source_ts:
    del source_ts
gc.collect()

display(
    {
        "input": input_summary,
        "input_was_synthetic": input_was_synthetic,
        "synthetic": synthetic_summary,
        "synthetic_metadata": synthetic_result.metadata,
        "max_rss_gib": max_rss_gib(),
    }
)

Load input tree sequence: started | max_rss=0.15 GiB
Load input tree sequence: finished | elapsed=11.7s | max_rss=3.85 GiB
Building synthetic full ARG: started | max_rss=3.85 GiB
Building synthetic full ARG: running | elapsed=30.0s | max_rss=5.87 GiB
Building synthetic full ARG: running | elapsed=60.0s | max_rss=7.41 GiB
Building synthetic full ARG: running | elapsed=90.1s | max_rss=8.96 GiB
Building synthetic full ARG: running | elapsed=120.1s | max_rss=9.92 GiB
Building synthetic full ARG: running | elapsed=151.6s | max_rss=10.47 GiB
Building synthetic full ARG: running | elapsed=209.2s | max_rss=16.72 GiB
Building synthetic full ARG: running | elapsed=239.3s | max_rss=16.72 GiB
Building synthetic full ARG: running | elapsed=276.1s | max_rss=16.72 GiB
Building synthetic full ARG: finished | elapsed=386.6s | max_rss=16.72 GiB


{'input': {'sequence_length': 243199375.0,
  'trees': 2119205,
  'samples': 5008,
  'nodes': 2746299,
  'edges': 25239529,
  'sites': 3348286,
  'mutations': 3348286},
 'input_was_synthetic': False,
 'synthetic': {'sequence_length': 243199375.0,
  'trees': 2119205,
  'samples': 5008,
  'nodes': 47668563,
  'edges': 211819125,
  'sites': 3348286,
  'mutations': 3348286},
 'synthetic_metadata': {'source': 'synthetic_full_arg',
  'time_rule': 'midpoint',
  'split_rule': 'balanced',
  'ensure_unique_event_times': True,
  'event_times_are_globally_unique': True,
  'event_time_adjustment_rule': 'scale_aware_bidirectional_spacing',
  'event_time_adjusted_event_count': 24334769,
  'max_event_time_adjustment': 0.0013627550248205278,
  'original_num_nodes': 2746299,
  'original_num_edges': 25239529,
  'original_num_trees': 2119205,
  'synthetic_recombination_event_count': 22461132,
  'synthetic_recombination_node_count': 44922264,
  'augmented_num_nodes': 47668563,
  'augmented_num_edges': 21181

## Build the Fast Trace and Terminal State

The cursor processes every event exactly. `CURSOR_CHUNK_SIZE` only controls how many events compiled code handles before returning to Python.

In [5]:
trace = run_stage(
    "Build FastARGTrace",
    lambda: build_fast_trace_from_full_arg(
        synthetic_arg, require_unique_event_times=True
    ),
)

trace_summary = {
    "steps": int(trace.num_steps),
    "events": int(trace.event_count),
    "recombination_events": int(trace.recombination_event_count),
    "coalescence_events": int(trace.coalescence_event_count),
    "samples": int(trace.sample_nodes.size),
    "nodes": int(trace.node_time.size),
    "edges": int(trace.edge_parent.size),
}
assert trace.recombination_event_count == synthetic_result.metadata[
    "synthetic_recombination_event_count"
]
assert trace.node_time.size == synthetic_arg.num_nodes
assert trace.edge_parent.size == synthetic_arg.num_edges

def warm_cursor_kernels():
    warmup_state = trace.initial_state(chunk_size=1)
    if trace.num_steps:
        warmup_state.advance().backtrack()
    return warmup_state


warmup_state = run_stage("JIT warm-up for forward and backward kernels", warm_cursor_kernels)
del warmup_state
gc.collect()

display({"trace": trace_summary, "max_rss_gib": max_rss_gib()})

Build FastARGTrace: started | max_rss=16.72 GiB
Build FastARGTrace: finished | elapsed=17.6s | max_rss=16.72 GiB
JIT warm-up for forward and backward kernels: started | max_rss=16.72 GiB
JIT warm-up for forward and backward kernels: finished | elapsed=0.0s | max_rss=16.72 GiB


{'trace': {'steps': 25202423,
  'events': 25202423,
  'recombination_events': 22461132,
  'coalescence_events': 2741291,
  'samples': 5008,
  'nodes': 47668563,
  'edges': 211819125},
 'max_rss_gib': 16.720123291015625}

In [16]:
terminal_state = run_stage(
    "Initialize terminal cursor at step zero",
    lambda: trace.initial_state(chunk_size=CURSOR_CHUNK_SIZE),
)

Initialize terminal cursor at step zero: started | max_rss=16.72 GiB
Initialize terminal cursor at step zero: finished | elapsed=0.0s | max_rss=16.72 GiB


In [17]:
def move_with_progress(state, target_step, label, progress_events=PROGRESS_EVENTS):
    target_step = int(target_step)
    if target_step < 0 or target_step > state.trace.num_steps:
        raise ValueError(f"target_step must be between 0 and {state.trace.num_steps}")

    origin = int(state.step)
    total = abs(target_step - origin)
    started = time.perf_counter()
    if total == 0:
        emit(f"{label}: already at step {target_step}")
        return state

    report_span = total if progress_events <= 0 else int(progress_events)
    while state.step != target_step:
        if target_step > state.step:
            next_step = min(target_step, state.step + report_span)
        else:
            next_step = max(target_step, state.step - report_span)
        state.move_to(next_step)

        moved = abs(int(state.step) - origin)
        elapsed = time.perf_counter() - started
        rate = moved / elapsed if elapsed else float("inf")
        emit(
            f"{label}: step={state.step:,}/{state.trace.num_steps:,} "
            f"movement={moved:,}/{total:,} ({100.0 * moved / total:.2f}%) "
            f"elapsed={elapsed:.1f}s rate={rate:,.0f} events/s "
            f"active={state.active_count:,} segments={state.segment_count:,} "
            f"max_rss={max_rss_gib():.2f} GiB"
        )
    return state

terminal_state = run_stage(
    "Initialize terminal cursor at step zero",
    lambda: trace.initial_state(chunk_size=CURSOR_CHUNK_SIZE),
)
move_with_progress(terminal_state, trace.num_steps, "Construct terminal state")

assert terminal_state.is_terminal
assert terminal_state.step == trace.num_steps
assert terminal_state.visible_node_ids.size == synthetic_arg.num_nodes
assert terminal_state.visible_edge_ids.size == synthetic_arg.num_edges

display(state_summary(terminal_state))
terminal_state

Initialize terminal cursor at step zero: started | max_rss=16.72 GiB
Initialize terminal cursor at step zero: finished | elapsed=0.0s | max_rss=16.72 GiB
Construct terminal state: step=1,000,000/25,202,423 movement=1,000,000/25,202,423 (3.97%) elapsed=2.2s rate=451,907 events/s active=1,005,008 segments=1,005,008 max_rss=16.72 GiB
Construct terminal state: step=2,000,000/25,202,423 movement=2,000,000/25,202,423 (7.94%) elapsed=2.5s rate=809,762 events/s active=2,005,008 segments=2,005,008 max_rss=16.72 GiB
Construct terminal state: step=3,000,000/25,202,423 movement=3,000,000/25,202,423 (11.90%) elapsed=2.6s rate=1,156,610 events/s active=3,005,008 segments=3,005,008 max_rss=16.72 GiB
Construct terminal state: step=4,000,000/25,202,423 movement=4,000,000/25,202,423 (15.87%) elapsed=2.7s rate=1,482,122 events/s active=4,005,008 segments=4,005,008 max_rss=16.72 GiB
Construct terminal state: step=5,000,000/25,202,423 movement=5,000,000/25,202,423 (19.84%) elapsed=2.8s rate=1,785,002 event

{'step': 25202423,
 'num_steps': 25202423,
 'current_time': 5006.0,
 'is_terminal': True,
 'visible_nodes': 47668563,
 'visible_edges': 211819125,
 'active_lineages': 12048,
 'active_segments': 75353}

In [18]:
move_with_progress(terminal_state, trace.num_steps, "Construct terminal state")

assert terminal_state.is_terminal
assert terminal_state.step == trace.num_steps
assert terminal_state.visible_node_ids.size == synthetic_arg.num_nodes
assert terminal_state.visible_edge_ids.size == synthetic_arg.num_edges

display(state_summary(terminal_state))
terminal_state

Construct terminal state: already at step 25202423


{'step': 25202423,
 'num_steps': 25202423,
 'current_time': 5006.0,
 'is_terminal': True,
 'visible_nodes': 47668563,
 'visible_edges': 211819125,
 'active_lineages': 12048,
 'active_segments': 75353}

In [25]:
trace.num_steps

25202423

In [11]:
terminal_state.trace.event_at_index(0)

ARGEvent(step=1, kind='recombination', time=0.07142741632958828, node_ids=(5000295, 5000296), edge_ids=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 2

In [ ]:
terminal_state.trace.event_at_index(trace.event_count - 1) if trace.event_count else None

## Trace Back to Any Partial State

`terminal_state` remains at the complete ARG. The reusable `traceback_state` is the cursor to move backward or forward. A step is an exact event boundary; a time query selects all events at or before that time.

In [8]:
traceback_state = run_stage("Clone terminal state for traceback", terminal_state.clone)

def step_at_or_before_time(target_time):
    target_time = float(target_time)
    if not np.isfinite(target_time):
        raise ValueError("target_time must be finite")
    return int(np.searchsorted(trace.event_time, target_time, side="right"))


def move_traceback_to_step(step):
    return move_with_progress(traceback_state, int(step), "Move traceback cursor")


def move_traceback_to_time(target_time):
    step = step_at_or_before_time(target_time)
    emit(f"time={float(target_time):.6g} maps to completed step={step:,}")
    return move_traceback_to_step(step)


def return_traceback_to_terminal():
    return move_with_progress(
        traceback_state,
        trace.num_steps,
        "Return traceback cursor to terminal",
    )


def frontier_preview(frontier, max_lineages=12):
    limit = min(int(max_lineages), len(frontier))
    return [
        {
            "node_id": int(frontier.node_ids[index]),
            "segments": frontier.segments_for_index(index),
        }
        for index in range(limit)
    ]

Clone terminal state for traceback: started | max_rss=16.72 GiB
Clone terminal state for traceback: finished | elapsed=0.0s | max_rss=16.72 GiB


In [9]:
TRACEBACK_STEP = max(0, trace.num_steps - min(TRACEBACK_EVENTS, trace.num_steps))
move_traceback_to_step(TRACEBACK_STEP)
partial_frontier = run_stage(
    "Materialize exact active frontier",
    traceback_state.compact_active_frontier,
)

display(
    {
        "partial_state": state_summary(traceback_state),
        "frontier_lineages": len(partial_frontier),
        "frontier_segments": partial_frontier.segment_count,
        "lineage_preview": frontier_preview(partial_frontier),
    }
)
traceback_state

Move traceback cursor: step=1,000,000/25,202,423 movement=1,000,000/25,192,423 (3.97%) elapsed=2.5s rate=404,472 events/s active=1,005,008 segments=1,005,008 max_rss=16.72 GiB
Move traceback cursor: step=2,000,000/25,202,423 movement=2,000,000/25,192,423 (7.94%) elapsed=2.8s rate=719,233 events/s active=2,005,008 segments=2,005,008 max_rss=16.72 GiB
Move traceback cursor: step=3,000,000/25,202,423 movement=3,000,000/25,192,423 (11.91%) elapsed=2.9s rate=1,023,109 events/s active=3,005,008 segments=3,005,008 max_rss=16.72 GiB
Move traceback cursor: step=4,000,000/25,202,423 movement=4,000,000/25,192,423 (15.88%) elapsed=3.1s rate=1,309,024 events/s active=4,005,008 segments=4,005,008 max_rss=16.72 GiB
Move traceback cursor: step=5,000,000/25,202,423 movement=5,000,000/25,192,423 (19.85%) elapsed=3.2s rate=1,574,966 events/s active=5,005,008 segments=5,005,008 max_rss=16.72 GiB
Move traceback cursor: step=6,000,000/25,202,423 movement=6,000,000/25,192,423 (23.82%) elapsed=3.3s rate=1,823

{'partial_state': {'step': 25192423,
  'num_steps': 25202423,
  'current_time': 5003.307931452499,
  'is_terminal': False,
  'visible_nodes': 47651696,
  'visible_edges': 210541967,
  'active_lineages': 1220051,
  'active_segments': 1243035},
 'frontier_lineages': 1220051,
 'frontier_segments': 1243035,
 'lineage_preview': [{'node_id': 3218971,
   'segments': ((34533007.0, 34533045.0),)},
  {'node_id': 3333273, 'segments': ((12987940.0, 12988858.0),)},
  {'node_id': 3644611, 'segments': ((239893010.0, 239894846.0),)},
  {'node_id': 3929247, 'segments': ((149278179.0, 149278629.0),)},
  {'node_id': 3943675, 'segments': ((116094177.0, 116094213.0),)},
  {'node_id': 4164195, 'segments': ((33127786.0, 33127796.0),)},
  {'node_id': 4193913, 'segments': ((120725861.0, 120730025.0),)},
  {'node_id': 4199809, 'segments': ((44292520.0, 44292982.0),)},
  {'node_id': 4201809, 'segments': ((209900939.0, 209905406.0),)},
  {'node_id': 6536117, 'segments': ((239546769.0, 239546821.0),)},
  {'node_id

## Marginal Trees from a State

This helper builds topology tables directly from the trace and does not reload the input file. The demonstration is windowed. Full-chromosome construction is available only with `allow_full=True` because terminal chr2 contains hundreds of millions of edges. Sites and mutations are not copied into these topology-only state snapshots.

In [ ]:
def marginal_tree_sequence_for_state(
    state,
    genomic_range=None,
    *,
    allow_full=False,
):
    if not isinstance(state, FastARGState):
        raise TypeError("state must be a FastARGState")
    if state.trace is not trace:
        raise ValueError("state belongs to a different FastARGTrace")
    if genomic_range is None and not allow_full:
        raise ValueError(
            "Full-chromosome materialization is disabled by default; pass a "
            "genomic_range or set allow_full=True explicitly"
        )
    return trace.to_tree_sequence_at_step(
        state.step,
        genomic_range=genomic_range,
    )


MARGINAL_WINDOW = (0.0, min(1_000_000.0, float(trace.sequence_length)))
terminal_window_ts = run_stage(
    f"Build terminal marginal trees for window {MARGINAL_WINDOW}",
    lambda: marginal_tree_sequence_for_state(
        terminal_state,
        genomic_range=MARGINAL_WINDOW,
    ),
)

display(
    {
        "window": MARGINAL_WINDOW,
        "trees_in_table_sequence": int(terminal_window_ts.num_trees),
        "nodes": int(terminal_window_ts.num_nodes),
        "edges": int(terminal_window_ts.num_edges),
    }
)

## Round-Trip and Optional Marginal Validation

The round-trip check always runs. Exact marginal-tree validation is intended for a small fixture and is enabled with `ARG_VERIFY_MARGINALS=1`; it suppresses inserted unary recombination paths when comparing the replayed terminal trees with the input trees.

In [ ]:
def assert_frontiers_equal(left, right):
    assert np.array_equal(left.node_ids, right.node_ids)
    assert np.array_equal(left.segment_offsets, right.segment_offsets)
    assert np.array_equal(left.segment_left, right.segment_left)
    assert np.array_equal(left.segment_right, right.segment_right)


def assert_same_marginal_genealogies(source, replayed, original_num_nodes):
    if source.sequence_length != replayed.sequence_length:
        raise AssertionError("sequence lengths differ")
    if source.num_trees != replayed.num_trees:
        raise AssertionError("marginal interval counts differ")

    replayed_flags = np.asarray(replayed.nodes_flags, dtype=np.uint32)
    compared_trees = 0
    for tree_index, (source_tree, replayed_tree) in enumerate(
        zip(source.trees(), replayed.trees())
    ):
        source_interval = (source_tree.interval.left, source_tree.interval.right)
        replayed_interval = (replayed_tree.interval.left, replayed_tree.interval.right)
        if source_interval != replayed_interval:
            raise AssertionError(
                f"tree {tree_index} intervals differ: "
                f"{source_interval} != {replayed_interval}"
            )

        for node_id in source_tree.nodes():
            source_parent = int(source_tree.parent(node_id))
            replayed_parent = int(replayed_tree.parent(node_id))
            while replayed_parent != tskit.NULL and replayed_parent >= original_num_nodes:
                if not (int(replayed_flags[replayed_parent]) & NODE_IS_RE_EVENT):
                    raise AssertionError(
                        f"tree {tree_index} inserted non-recombination node "
                        f"{replayed_parent} on the path from {node_id}"
                    )
                if replayed_tree.num_children(replayed_parent) != 1:
                    raise AssertionError(
                        f"tree {tree_index} recombination node {replayed_parent} "
                        "is not unary in the marginal tree"
                    )
                replayed_parent = int(replayed_tree.parent(replayed_parent))
            if source_parent != replayed_parent:
                raise AssertionError(
                    f"tree {tree_index} parent differs for node {node_id}: "
                    f"{source_parent} != {replayed_parent}"
                )
        compared_trees += 1
    return compared_trees


terminal_frontier = run_stage(
    "Materialize reference terminal frontier",
    terminal_state.compact_active_frontier,
)
return_traceback_to_terminal()
traceback_terminal_frontier = run_stage(
    "Materialize round-trip terminal frontier",
    traceback_state.compact_active_frontier,
)
assert_frontiers_equal(terminal_frontier, traceback_terminal_frontier)
move_traceback_to_step(TRACEBACK_STEP)
assert_frontiers_equal(partial_frontier, traceback_state.compact_active_frontier())
assert terminal_state.is_terminal

compared_marginal_trees = None
if VERIFY_MARGINALS:
    terminal_replayed_ts = run_stage(
        "Materialize complete terminal topology for validation",
        lambda: trace.to_tree_sequence_at_step(trace.num_steps),
    )
    compared_marginal_trees = run_stage(
        "Compare input and terminal marginal genealogies",
        lambda: assert_same_marginal_genealogies(
            validation_source_ts,
            terminal_replayed_ts,
            input_summary["nodes"],
        ),
    )
    del terminal_replayed_ts
else:
    emit("Exact marginal validation skipped; set ARG_VERIFY_MARGINALS=1 for a small input.")

validation_summary = {
    "terminal_round_trip_exact": True,
    "partial_round_trip_exact": True,
    "terminal_state_preserved": bool(terminal_state.is_terminal),
    "traceback_step": int(traceback_state.step),
    "compared_marginal_trees": compared_marginal_trees,
    "max_rss_gib": max_rss_gib(),
}
validation_summary

## Multiple Strictly Region-Separable Boundaries

This section builds a separate catalog at every configured boundary. At a cut, overlapping half-open frontier material is connected, disjoint material carried by one lineage remains connected, and later suffix events merge any components they couple. The final union-find roots are therefore the minimal structural closures whose outside nodes and events can remain fixed.

Set `ARG_SEPARABLE_BOUNDARIES` to comma-separated selectors such as `step:100`, `time:25.0`, `event:99`, or `node:1234`. Event indices are zero-based; event and node selectors cut immediately before the associated event; time selectors include every event at or before the requested time. Normalized duplicate steps are combined and scanned from oldest to youngest.

Noncontiguous closures and whole-sequence closures are retained in the full catalog but are not eligible local regions. This section detects structural separability only: it does not score badness or apply ARG actions.


In [19]:
from numba import njit


SEPARABLE_BOUNDARY_TEXT = os.environ.get(
    "ARG_SEPARABLE_BOUNDARIES",
    f"step:{TRACEBACK_STEP}",
)
SEPARABLE_MAX_SUFFIX_EVENTS = int(
    os.environ.get("ARG_SEPARABLE_MAX_SUFFIX_EVENTS", "1000000")
)
SEPARABLE_ALLOW_LARGE_SCAN = environment_flag(
    "ARG_SEPARABLE_ALLOW_LARGE_SCAN",
    False,
)
SEPARABLE_MAX_DISPLAY = int(os.environ.get("ARG_SEPARABLE_MAX_DISPLAY", "50"))

if SEPARABLE_MAX_SUFFIX_EVENTS < 0:
    raise ValueError("ARG_SEPARABLE_MAX_SUFFIX_EVENTS must be nonnegative")
if SEPARABLE_MAX_DISPLAY < 0:
    raise ValueError("ARG_SEPARABLE_MAX_DISPLAY must be nonnegative")


def normalize_separable_boundary_specs(text, trace):
    """Normalize comma-separated step/time/event/node selectors to unique cuts."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("ARG_SEPARABLE_BOUNDARIES must contain at least one selector")

    by_step = {}
    for raw_selector in text.split(","):
        selector = raw_selector.strip()
        if not selector or ":" not in selector:
            raise ValueError(
                f"malformed boundary selector {raw_selector!r}; expected kind:value"
            )
        kind, value_text = (part.strip() for part in selector.split(":", 1))
        kind = kind.lower()
        if not value_text:
            raise ValueError(f"boundary selector {selector!r} has no value")

        if kind == "step":
            try:
                step = trace._validate_step(int(value_text))
            except (TypeError, ValueError) as error:
                raise ValueError(f"invalid step selector {selector!r}") from error
        elif kind == "time":
            try:
                requested_time = float(value_text)
            except ValueError as error:
                raise ValueError(f"invalid time selector {selector!r}") from error
            if not np.isfinite(requested_time):
                raise ValueError(f"time selector must be finite: {selector!r}")
            step = int(np.searchsorted(trace.event_time, requested_time, side="right"))
        elif kind == "event":
            try:
                event_index = int(value_text)
            except ValueError as error:
                raise ValueError(f"invalid event selector {selector!r}") from error
            if event_index < 0 or event_index >= trace.event_count:
                raise ValueError(
                    f"event index must be in [0, {trace.event_count}), got {event_index}"
                )
            step = event_index
        elif kind == "node":
            try:
                node_id = int(value_text)
            except ValueError as error:
                raise ValueError(f"invalid node selector {selector!r}") from error
            if node_id < 0 or node_id >= trace.node_reveal_step.size:
                raise ValueError(
                    f"node id must be in [0, {trace.node_reveal_step.size}), got {node_id}"
                )
            reveal_step = int(trace.node_reveal_step[node_id])
            if reveal_step <= 0:
                raise ValueError(
                    f"node:{node_id} is a sample or is never revealed by an event"
                )
            step = reveal_step - 1
        else:
            raise ValueError(
                f"unknown boundary selector kind {kind!r}; "
                "expected step, time, event, or node"
            )

        entry = by_step.setdefault(
            int(step),
            {"step": int(step), "selectors": [], "selector_kinds": []},
        )
        entry["selectors"].append(selector)
        entry["selector_kinds"].append(kind)

    normalized = []
    for step in sorted(by_step, reverse=True):
        entry = by_step[step]
        entry["selectors"] = tuple(entry["selectors"])
        entry["selector_kinds"] = tuple(dict.fromkeys(entry["selector_kinds"]))
        entry["time"] = 0.0 if step == 0 else float(trace.event_time[step - 1])
        normalized.append(entry)
    return normalized


separable_boundary_specs = normalize_separable_boundary_specs(
    SEPARABLE_BOUNDARY_TEXT,
    trace,
)

display(
    {
        "configured": SEPARABLE_BOUNDARY_TEXT,
        "normalized_boundaries_oldest_to_youngest": separable_boundary_specs,
        "max_suffix_events": SEPARABLE_MAX_SUFFIX_EVENTS,
        "allow_large_scan": SEPARABLE_ALLOW_LARGE_SCAN,
        "display_limit": SEPARABLE_MAX_DISPLAY,
    }
)


{'configured': 'step:25192423',
 'normalized_boundaries_oldest_to_youngest': [{'step': 25192423,
   'selectors': ('step:25192423',),
   'selector_kinds': ('step',),
   'time': 5003.307931452499}],
 'max_suffix_events': 1000000,
 'allow_large_scan': False,
 'display_limit': 50}

In [20]:
@njit
def _separable_find(parent, item):
    root = item
    while parent[root] != root:
        root = parent[root]
    while parent[item] != item:
        next_item = parent[item]
        parent[item] = root
        item = next_item
    return root


@njit
def _separable_union(parent, sizes, left_item, right_item):
    left_root = _separable_find(parent, left_item)
    right_root = _separable_find(parent, right_item)
    if left_root == right_root:
        return left_root
    if sizes[left_root] < sizes[right_root]:
        left_root, right_root = right_root, left_root
    parent[right_root] = left_root
    sizes[left_root] += sizes[right_root]
    return left_root


@njit
def _initial_material_components_kernel(
    segment_offsets,
    segment_left,
    segment_right,
    left_order,
):
    lineage_count = segment_offsets.size - 1
    parent = np.arange(lineage_count, dtype=np.int32)
    sizes = np.ones(lineage_count, dtype=np.int32)
    segment_owner = np.empty(segment_left.size, dtype=np.int32)

    for lineage_index in range(lineage_count):
        for segment_index in range(
            segment_offsets[lineage_index],
            segment_offsets[lineage_index + 1],
        ):
            segment_owner[segment_index] = lineage_index

    run_owner = -1
    run_right = 0.0
    for order_index in range(left_order.size):
        segment_index = left_order[order_index]
        owner = segment_owner[segment_index]
        left = segment_left[segment_index]
        right = segment_right[segment_index]
        if run_owner < 0 or left >= run_right:
            run_owner = owner
            run_right = right
        else:
            _separable_union(parent, sizes, run_owner, owner)
            if right > run_right:
                run_right = right

    lineage_roots = np.empty(lineage_count, dtype=np.int32)
    for lineage_index in range(lineage_count):
        lineage_roots[lineage_index] = _separable_find(parent, lineage_index)
    return parent, sizes, segment_owner, lineage_roots


@njit
def _close_components_through_suffix_kernel(
    frontier_node_ids,
    cut_step,
    event_node1,
    event_node2,
    event_edge_start,
    revealed_edge_ids,
    edge_child,
    node_count,
    parent,
    sizes,
):
    node_component = np.full(node_count, -1, dtype=np.int32)
    for lineage_index in range(frontier_node_ids.size):
        node_component[frontier_node_ids[lineage_index]] = lineage_index

    suffix_event_count = event_node1.size - cut_step
    event_roots = np.full(suffix_event_count, -1, dtype=np.int32)
    bad_event = -1
    bad_child = -1

    for event_index in range(cut_step, event_node1.size):
        edge_start = event_edge_start[event_index]
        edge_end = event_edge_start[event_index + 1]
        if edge_start == edge_end:
            continue

        event_root = -1
        for edge_position in range(edge_start, edge_end):
            edge_id = revealed_edge_ids[edge_position]
            child_id = edge_child[edge_id]
            child_component = node_component[child_id]
            if child_component < 0:
                bad_event = event_index
                bad_child = child_id
                frontier_roots = np.empty(frontier_node_ids.size, dtype=np.int32)
                return (
                    parent,
                    sizes,
                    frontier_roots,
                    event_roots,
                    node_component,
                    bad_event,
                    bad_child,
                )
            if event_root < 0:
                event_root = _separable_find(parent, child_component)
            else:
                event_root = _separable_union(
                    parent,
                    sizes,
                    event_root,
                    child_component,
                )

        node1 = event_node1[event_index]
        node2 = event_node2[event_index]
        if node1 >= 0:
            previous = node_component[node1]
            if previous >= 0:
                event_root = _separable_union(parent, sizes, event_root, previous)
            node_component[node1] = event_root
        if node2 >= 0:
            previous = node_component[node2]
            if previous >= 0:
                event_root = _separable_union(parent, sizes, event_root, previous)
            node_component[node2] = event_root
        event_roots[event_index - cut_step] = event_root

    frontier_roots = np.empty(frontier_node_ids.size, dtype=np.int32)
    for lineage_index in range(frontier_node_ids.size):
        frontier_roots[lineage_index] = _separable_find(parent, lineage_index)
    for event_offset in range(event_roots.size):
        if event_roots[event_offset] >= 0:
            event_roots[event_offset] = _separable_find(
                parent,
                event_roots[event_offset],
            )
    return (
        parent,
        sizes,
        frontier_roots,
        event_roots,
        node_component,
        bad_event,
        bad_child,
    )


@njit
def _component_labels_for_nodes_kernel(node_ids, node_component, parent):
    labels = np.full(node_ids.size, -1, dtype=np.int32)
    for index in range(node_ids.size):
        component = node_component[node_ids[index]]
        if component >= 0:
            labels[index] = _separable_find(parent, component)
    return labels


@njit
def _merge_component_intervals_kernel(
    component_count,
    segment_component,
    segment_left,
    segment_right,
    component_left_order,
):
    merged_left = np.empty(segment_left.size, dtype=np.float64)
    merged_right = np.empty(segment_right.size, dtype=np.float64)
    merged_offsets = np.empty(component_count + 1, dtype=np.int64)
    output_size = 0
    order_position = 0

    for component in range(component_count):
        merged_offsets[component] = output_size
        has_interval = False
        current_left = 0.0
        current_right = 0.0
        while (
            order_position < component_left_order.size
            and segment_component[component_left_order[order_position]] == component
        ):
            segment_index = component_left_order[order_position]
            left = segment_left[segment_index]
            right = segment_right[segment_index]
            if not has_interval:
                current_left = left
                current_right = right
                has_interval = True
            elif left <= current_right:
                if right > current_right:
                    current_right = right
            else:
                merged_left[output_size] = current_left
                merged_right[output_size] = current_right
                output_size += 1
                current_left = left
                current_right = right
            order_position += 1
        if has_interval:
            merged_left[output_size] = current_left
            merged_right[output_size] = current_right
            output_size += 1

    merged_offsets[component_count] = output_size
    return (
        merged_offsets,
        merged_left[:output_size],
        merged_right[:output_size],
    )


In [21]:
def initial_material_components(segment_offsets, segment_left, segment_right):
    segment_offsets = np.asarray(segment_offsets, dtype=np.int64)
    segment_left = np.asarray(segment_left, dtype=np.float64)
    segment_right = np.asarray(segment_right, dtype=np.float64)
    if segment_offsets.ndim != 1 or segment_offsets.size == 0:
        raise ValueError("segment_offsets must be a nonempty one-dimensional array")
    if segment_left.shape != segment_right.shape or segment_left.ndim != 1:
        raise ValueError("segment_left and segment_right must be matching vectors")
    if int(segment_offsets[0]) != 0 or int(segment_offsets[-1]) != segment_left.size:
        raise ValueError("segment_offsets must span the segment arrays")
    if np.any(np.diff(segment_offsets) < 0):
        raise ValueError("segment_offsets must be nondecreasing")
    if np.any(segment_left >= segment_right):
        raise ValueError("all material intervals must be nonempty")

    left_order = np.argsort(segment_left, kind="stable").astype(np.int64)
    parent, sizes, segment_owner, lineage_roots = (
        _initial_material_components_kernel(
            segment_offsets,
            segment_left,
            segment_right,
            left_order,
        )
    )
    return {
        "parent": parent,
        "sizes": sizes,
        "segment_owner": segment_owner,
        "lineage_roots": lineage_roots,
    }


def canonical_component_intervals(
    component_count,
    segment_component,
    segment_left,
    segment_right,
):
    segment_component = np.asarray(segment_component, dtype=np.int32)
    segment_left = np.asarray(segment_left, dtype=np.float64)
    segment_right = np.asarray(segment_right, dtype=np.float64)
    order = np.lexsort((segment_right, segment_left, segment_component)).astype(
        np.int64
    )
    offsets, merged_left, merged_right = _merge_component_intervals_kernel(
        int(component_count),
        segment_component,
        segment_left,
        segment_right,
        order,
    )
    return [
        tuple(
            (float(left), float(right))
            for left, right in zip(
                merged_left[offsets[index] : offsets[index + 1]],
                merged_right[offsets[index] : offsets[index + 1]],
            )
        )
        for index in range(int(component_count))
    ]


def group_values_by_component(values, components, component_count):
    values = np.asarray(values)
    components = np.asarray(components, dtype=np.int32)
    if values.size != components.size:
        raise ValueError("values and components must have matching lengths")
    buckets = [[] for _ in range(int(component_count))]
    for value, component in zip(values, components):
        component = int(component)
        if component < 0 or component >= component_count:
            raise AssertionError(f"unassigned component label {component}")
        buckets[component].append(int(value))
    return tuple(tuple(bucket) for bucket in buckets)


def region_rejection_reasons(intervals, sequence_length):
    reasons = []
    if len(intervals) != 1:
        reasons.append("noncontiguous")
    if (
        len(intervals) == 1
        and intervals[0][0] <= 0.0
        and intervals[0][1] >= float(sequence_length)
    ):
        reasons.append("whole_sequence")
    return tuple(reasons)


def assert_candidate_regions_do_not_overlap(candidates):
    labeled_intervals = sorted(
        (
            float(left),
            float(right),
            candidate["candidate_id"],
        )
        for candidate in candidates
        for left, right in candidate["intervals"]
    )
    active_right = -np.inf
    active_candidate = None
    for left, right, candidate_id in labeled_intervals:
        if left < active_right and candidate_id != active_candidate:
            raise AssertionError(
                f"closure regions overlap: {active_candidate} and {candidate_id}"
            )
        if right > active_right:
            active_right = right
            active_candidate = candidate_id


def strict_separable_regions_at_boundary(state, boundary, terminal_frontier):
    """Return all minimal suffix-closed components at one already-positioned cut."""
    state_trace = state.trace
    cut_step = int(boundary["step"])
    if state.step != cut_step:
        raise ValueError(f"state is at step {state.step}, expected cut {cut_step}")

    frontier = state.compact_active_frontier()
    if frontier.node_ids.size == 0 or frontier.segment_left.size == 0:
        raise RuntimeError(f"boundary step {cut_step} has an empty active frontier")

    initial = initial_material_components(
        frontier.segment_offsets,
        frontier.segment_left,
        frontier.segment_right,
    )
    (
        parent,
        sizes,
        frontier_roots,
        event_roots,
        node_component,
        bad_event,
        bad_child,
    ) = _close_components_through_suffix_kernel(
        frontier.node_ids,
        cut_step,
        state_trace.event_node1,
        state_trace.event_node2,
        state_trace.event_edge_start,
        state_trace.revealed_edge_ids,
        state_trace.edge_child,
        state_trace.node_time.size,
        initial["parent"],
        initial["sizes"],
    )
    if bad_event >= 0:
        return {
            "status": "failed_unassigned_material",
            "boundary_step": cut_step,
            "boundary_time": float(state.current_time),
            "selectors": tuple(boundary["selectors"]),
            "selector_kinds": tuple(boundary["selector_kinds"]),
            "suffix_event_count": int(state_trace.num_steps - cut_step),
            "edgeful_suffix_event_count": None,
            "frontier_lineage_count": int(frontier.node_ids.size),
            "frontier_segment_count": int(frontier.segment_count),
            "component_count": 0,
            "eligible_count": 0,
            "closure_verified": False,
            "failed_event_index": int(bad_event),
            "missing_child_node_id": int(bad_child),
            "candidates": [],
            "message": (
                f"edgeful event {bad_event} references child node "
                f"{bad_child} with no frontier component"
            ),
        }

    final_roots = np.unique(frontier_roots)
    root_to_component = np.full(frontier.node_ids.size, -1, dtype=np.int32)
    for component_index, root in enumerate(final_roots):
        root_to_component[int(root)] = component_index

    frontier_components = root_to_component[frontier_roots]
    if np.any(frontier_components < 0):
        raise AssertionError("every lower-frontier lineage must have one closure")

    event_components = np.full(event_roots.size, -1, dtype=np.int32)
    edgeful = np.diff(state_trace.event_edge_start[cut_step:]) > 0
    assigned = event_roots >= 0
    if event_roots.size:
        event_components[assigned] = root_to_component[event_roots[assigned]]
    if not np.array_equal(assigned, edgeful):
        raise AssertionError("every edgeful suffix event must have exactly one closure")
    if np.any(event_components[edgeful] < 0):
        raise AssertionError("edgeful suffix event has no final closure")

    terminal_roots = _component_labels_for_nodes_kernel(
        terminal_frontier.node_ids,
        node_component,
        parent,
    )
    if np.any(terminal_roots < 0):
        missing = terminal_frontier.node_ids[terminal_roots < 0]
        raise RuntimeError(
            f"terminal frontier nodes have no closure assignment: {missing[:10]}"
        )
    terminal_components = root_to_component[terminal_roots]
    if np.any(terminal_components < 0):
        raise AssertionError("terminal frontier contains an unknown closure root")

    segment_components = frontier_components[initial["segment_owner"]]
    intervals_by_component = canonical_component_intervals(
        final_roots.size,
        segment_components,
        frontier.segment_left,
        frontier.segment_right,
    )
    lower_nodes = group_values_by_component(
        frontier.node_ids,
        frontier_components,
        final_roots.size,
    )
    terminal_nodes = group_values_by_component(
        terminal_frontier.node_ids,
        terminal_components,
        final_roots.size,
    )
    suffix_event_indices = np.arange(
        cut_step,
        state_trace.num_steps,
        dtype=np.int64,
    )
    events_by_component = group_values_by_component(
        suffix_event_indices[edgeful],
        event_components[edgeful],
        final_roots.size,
    )

    candidates = []
    for component_index, intervals in enumerate(intervals_by_component):
        event_indices = np.asarray(
            events_by_component[component_index],
            dtype=np.int64,
        )
        if event_indices.size:
            mutable_parent_nodes = np.unique(
                np.concatenate(
                    (
                        state_trace.event_node1[event_indices],
                        state_trace.event_node2[event_indices],
                    )
                )
            )
            mutable_parent_nodes = mutable_parent_nodes[
                mutable_parent_nodes >= 0
            ].astype(np.int64, copy=False)
        else:
            mutable_parent_nodes = np.empty(0, dtype=np.int64)

        lower_anchor_nodes = tuple(lower_nodes[component_index])
        original_terminal_nodes = tuple(terminal_nodes[component_index])
        mutable_parent_node_ids = tuple(int(v) for v in mutable_parent_nodes)
        closure_event_indices = tuple(int(v) for v in event_indices)
        material_length = float(sum(right - left for left, right in intervals))
        span = (
            0.0
            if not intervals
            else float(intervals[-1][1] - intervals[0][0])
        )
        rejection_reasons = region_rejection_reasons(
            intervals,
            state_trace.sequence_length,
        )
        all_nodes = set(lower_anchor_nodes)
        all_nodes.update(mutable_parent_node_ids)
        all_nodes.update(original_terminal_nodes)

        candidates.append(
            {
                "boundary_step": cut_step,
                "boundary_time": float(state.current_time),
                "boundary_selectors": tuple(boundary["selectors"]),
                "boundary_selector_kinds": tuple(boundary["selector_kinds"]),
                "region_key": tuple(intervals),
                "intervals": tuple(intervals),
                "left": float(intervals[0][0]),
                "right": float(intervals[-1][1]),
                "material_length": material_length,
                "span": span,
                "contiguous": len(intervals) == 1,
                "lower_frontier_anchor_node_ids": lower_anchor_nodes,
                "original_suffix_event_indices": closure_event_indices,
                "mutable_parent_node_ids": mutable_parent_node_ids,
                "original_terminal_lineage_ids": original_terminal_nodes,
                "lineage_count": len(lower_anchor_nodes),
                "event_count": len(closure_event_indices),
                "node_count": len(all_nodes),
                "terminal_lineage_count": len(original_terminal_nodes),
                "closure_verified": True,
                "eligible": not rejection_reasons,
                "rejection_reasons": rejection_reasons,
            }
        )

    candidates.sort(key=lambda candidate: candidate["region_key"])
    for candidate_index, candidate in enumerate(candidates):
        candidate["candidate_id"] = (
            f"step-{cut_step}-region-{candidate_index:06d}"
        )

    assert sum(candidate["lineage_count"] for candidate in candidates) == len(
        frontier.node_ids
    )
    assert sum(candidate["event_count"] for candidate in candidates) == int(
        np.sum(edgeful)
    )
    assert_candidate_regions_do_not_overlap(candidates)
    for candidate in candidates:
        if candidate["eligible"]:
            assert candidate["closure_verified"]
            assert candidate["contiguous"]
            assert 0.0 <= candidate["left"] < candidate["right"]
            assert candidate["material_length"] < state_trace.sequence_length
            assert not candidate["rejection_reasons"]

    return {
        "status": "ok",
        "boundary_step": cut_step,
        "boundary_time": float(state.current_time),
        "selectors": tuple(boundary["selectors"]),
        "selector_kinds": tuple(boundary["selector_kinds"]),
        "suffix_event_count": int(state_trace.num_steps - cut_step),
        "edgeful_suffix_event_count": int(np.sum(edgeful)),
        "frontier_lineage_count": int(frontier.node_ids.size),
        "frontier_segment_count": int(frontier.segment_count),
        "component_count": len(candidates),
        "eligible_count": sum(candidate["eligible"] for candidate in candidates),
        "candidates": candidates,
    }


def separable_candidate_preview(candidate):
    return {
        "candidate_id": candidate["candidate_id"],
        "boundary_step": candidate["boundary_step"],
        "region_key": candidate["region_key"],
        "material_length": candidate["material_length"],
        "contiguous": candidate["contiguous"],
        "lineages": candidate["lineage_count"],
        "events": candidate["event_count"],
        "nodes": candidate["node_count"],
        "closure_verified": candidate["closure_verified"],
        "eligible": candidate["eligible"],
        "rejection_reasons": candidate["rejection_reasons"],
    }


In [22]:
def _expect_value_error(callback):
    try:
        callback()
    except ValueError:
        return
    raise AssertionError("expected ValueError")


# Two disjoint material components remain independent and locally eligible.
_disjoint = initial_material_components(
    np.array([0, 1, 2, 3, 4], dtype=np.int64),
    np.array([0.0, 0.0, 150.0, 150.0]),
    np.array([100.0, 100.0, 250.0, 250.0]),
)
assert np.unique(_disjoint["lineage_roots"]).size == 2
_disjoint_root_values = np.unique(_disjoint["lineage_roots"])
_disjoint_root_to_component = {
    int(root): index for index, root in enumerate(_disjoint_root_values)
}
_disjoint_lineage_components = np.array(
    [
        _disjoint_root_to_component[int(root)]
        for root in _disjoint["lineage_roots"]
    ],
    dtype=np.int32,
)
_disjoint_intervals = canonical_component_intervals(
    2,
    _disjoint_lineage_components[_disjoint["segment_owner"]],
    np.array([0.0, 0.0, 150.0, 150.0]),
    np.array([100.0, 100.0, 250.0, 250.0]),
)
assert all(
    not region_rejection_reasons(intervals, 300.0)
    for intervals in _disjoint_intervals
)

# Half-open intervals that only touch at an endpoint remain separate.
_touching = initial_material_components(
    np.array([0, 1, 2], dtype=np.int64),
    np.array([0.0, 100.0]),
    np.array([100.0, 200.0]),
)
assert np.unique(_touching["lineage_roots"]).size == 2

# One lineage carrying disjoint material stays one recorded, ineligible closure.
_disjoint_one_lineage = initial_material_components(
    np.array([0, 2], dtype=np.int64),
    np.array([0.0, 20.0]),
    np.array([10.0, 30.0]),
)
_disjoint_one_intervals = canonical_component_intervals(
    1,
    np.zeros(2, dtype=np.int32),
    np.array([0.0, 20.0]),
    np.array([10.0, 30.0]),
)
assert _disjoint_one_intervals == [((0.0, 10.0), (20.0, 30.0))]
assert region_rejection_reasons(_disjoint_one_intervals[0], 40.0) == (
    "noncontiguous",
)

# A later edgeful event couples two initial components into one closure.
_merge_initial = initial_material_components(
    np.array([0, 1, 2], dtype=np.int64),
    np.array([0.0, 100.0]),
    np.array([100.0, 200.0]),
)
(
    _merge_parent,
    _merge_sizes,
    _merge_frontier_roots,
    _merge_event_roots,
    _merge_node_component,
    _merge_bad_event,
    _merge_bad_child,
) = _close_components_through_suffix_kernel(
    np.array([10, 11], dtype=np.int32),
    0,
    np.array([12], dtype=np.int32),
    np.array([-1], dtype=np.int32),
    np.array([0, 2], dtype=np.int32),
    np.array([0, 1], dtype=np.int32),
    np.array([10, 11], dtype=np.int32),
    13,
    _merge_initial["parent"],
    _merge_initial["sizes"],
)
assert _merge_bad_event == -1 and _merge_bad_child == -1
assert np.unique(_merge_frontier_roots).size == 1
assert _merge_event_roots[0] == _merge_frontier_roots[0]
assert (
    _component_labels_for_nodes_kernel(
        np.array([12], dtype=np.int32),
        _merge_node_component,
        _merge_parent,
    )[0]
    == _merge_frontier_roots[0]
)

# Selector normalization: all selector types, deduplication, ordering, and errors.
assert normalize_separable_boundary_specs("step:0", trace)[0]["step"] == 0
assert len(normalize_separable_boundary_specs("step:0,event:0", trace)) == 1
assert normalize_separable_boundary_specs("event:0", trace)[0]["step"] == 0
_expected_time_step = int(
    np.searchsorted(trace.event_time, trace.event_time[0], side="right")
)
assert (
    normalize_separable_boundary_specs(
        f"time:{trace.event_time[0]}",
        trace,
    )[0]["step"]
    == _expected_time_step
)
_event_node_ids = np.flatnonzero(trace.node_reveal_step > 0)
assert _event_node_ids.size
_event_node_id = int(_event_node_ids[0])
assert (
    normalize_separable_boundary_specs(f"node:{_event_node_id}", trace)[0][
        "step"
    ]
    == int(trace.node_reveal_step[_event_node_id]) - 1
)
_ordered = normalize_separable_boundary_specs(
    f"step:0,step:{trace.num_steps}",
    trace,
)
assert [entry["step"] for entry in _ordered] == [trace.num_steps, 0]

_expect_value_error(lambda: normalize_separable_boundary_specs("", trace))
_expect_value_error(
    lambda: normalize_separable_boundary_specs(
        f"step:{trace.num_steps + 1}",
        trace,
    )
)
_expect_value_error(
    lambda: normalize_separable_boundary_specs(
        f"event:{trace.event_count}",
        trace,
    )
)
_expect_value_error(
    lambda: normalize_separable_boundary_specs(
        f"node:{int(trace.sample_nodes[0])}",
        trace,
    )
)
_expect_value_error(
    lambda: normalize_separable_boundary_specs("not-a-selector", trace)
)
_expect_value_error(
    lambda: normalize_separable_boundary_specs("unknown:1", trace)
)

helper_validation_summary = {
    "two_independent_regions": "passed",
    "later_event_merge": "passed",
    "disjoint_one_lineage_ineligible": "passed",
    "half_open_endpoint_touch": "passed",
    "selector_normalization": "passed",
    "selector_validation": "passed",
}
display(helper_validation_summary)


{'two_independent_regions': 'passed',
 'later_event_merge': 'passed',
 'disjoint_one_lineage_ineligible': 'passed',
 'half_open_endpoint_touch': 'passed',
 'selector_normalization': 'passed',
 'selector_validation': 'passed'}

In [23]:
terminal_state.is_terminal

True

In [ ]:
terminal_step_before_separable_scan = terminal_state.step
traceback_step_before_separable_scan = traceback_state.step
assert terminal_state.is_terminal
terminal_frontier_for_separability = terminal_state.compact_active_frontier()
separable_scan_state = terminal_state.clone()

separable_boundary_results = []
for boundary in separable_boundary_specs:
    cut_step = int(boundary["step"])
    suffix_event_count = trace.num_steps - cut_step
    if (
        suffix_event_count > SEPARABLE_MAX_SUFFIX_EVENTS
        and not SEPARABLE_ALLOW_LARGE_SCAN
    ):
        separable_boundary_results.append(
            {
                "status": "skipped_large_suffix",
                "boundary_step": cut_step,
                "boundary_time": float(boundary["time"]),
                "selectors": tuple(boundary["selectors"]),
                "selector_kinds": tuple(boundary["selector_kinds"]),
                "suffix_event_count": int(suffix_event_count),
                "max_suffix_events": SEPARABLE_MAX_SUFFIX_EVENTS,
                "component_count": 0,
                "eligible_count": 0,
                "candidates": [],
                "message": (
                    "suffix exceeds the configured scan guard; set "
                    "ARG_SEPARABLE_ALLOW_LARGE_SCAN=1 to run it"
                ),
            }
        )
        continue

    move_with_progress(
        separable_scan_state,
        cut_step,
        f"Move separability scan state to step {cut_step}",
    )
    result = run_stage(
        f"Strict separability closure at step {cut_step}",
        lambda: strict_separable_regions_at_boundary(
            separable_scan_state,
            boundary,
            terminal_frontier_for_separability,
        ),
    )
    separable_boundary_results.append(result)

separable_region_candidates = [
    candidate
    for boundary_result in separable_boundary_results
    for candidate in boundary_result["candidates"]
]
eligible_separable_regions = [
    candidate
    for candidate in separable_region_candidates
    if candidate["eligible"]
    
]

assert terminal_state.step == terminal_step_before_separable_scan
assert terminal_state.is_terminal
assert traceback_state.step == traceback_step_before_separable_scan

_boundary_summary = [
    {
        "status": result["status"],
        "boundary_step": result["boundary_step"],
        "selectors": result["selectors"],
        "suffix_events": result["suffix_event_count"],
        "components": result["component_count"],
        "eligible": result["eligible_count"],
    }
    for result in separable_boundary_results
]
display(_boundary_summary)

_preview_count = min(SEPARABLE_MAX_DISPLAY, len(separable_region_candidates))
display(
    {
        "stored_candidate_count": len(separable_region_candidates),
        "stored_eligible_count": len(eligible_separable_regions),
        "displayed_candidate_count": _preview_count,
        "display_limit": SEPARABLE_MAX_DISPLAY,
        "candidates": [
            separable_candidate_preview(candidate)
            for candidate in separable_region_candidates[:_preview_count]
        ],
    }
)


Move separability scan state to step 25192423: step=25,192,423/25,202,423 movement=10,000/10,000 (100.00%) elapsed=0.4s rate=27,881 events/s active=1,220,051 segments=1,243,035 max_rss=16.72 GiB
Strict separability closure at step 25192423: started | max_rss=16.72 GiB
Strict separability closure at step 25192423: finished | elapsed=1.9s | max_rss=16.72 GiB


[{'status': 'ok',
  'boundary_step': 25192423,
  'selectors': ('step:25192423',),
  'suffix_events': 10000,
  'components': 7367,
  'eligible': 3411}]

{'stored_candidate_count': 7367,
 'stored_eligible_count': 3411,
 'displayed_candidate_count': 50,
 'display_limit': 50,
 'candidates': [{'candidate_id': 'step-25192423-region-000000',
   'boundary_step': 25192423,
   'region_key': ((0.0, 51228.0),
    (52111.0, 52175.0),
    (52734.0, 104961.0),
    (105517.0, 109073.0),
    (110653.0, 110879.0),
    (111486.0, 143559.0),
    (143731.0, 153280.0),
    (153314.0, 156145.0),
    (156773.0, 163537.0),
    (164146.0, 199422.0),
    (199446.0, 298309.0),
    (301052.0, 324841.0),
    (325671.0, 335003.0),
    (335702.0, 355300.0),
    (355354.0, 358422.0),
    (359211.0, 366012.0),
    (366384.0, 447520.0),
    (450533.0, 467552.0),
    (467741.0, 469833.0),
    (470702.0, 522245.0),
    (522330.0, 524816.0),
    (524886.0, 559081.0),
    (559805.0, 564242.0),
    (564263.0, 564273.0),
    (564770.0, 565103.0),
    (566810.0, 576187.0),
    (577116.0, 577186.0),
    (577417.0, 579293.0),
    (579393.0, 580340.0),
    (580800.0, 584350.0),


In [32]:
actionable_separable_regions = [
    candidate
    for candidate in eligible_separable_regions
    if candidate["event_count"] > 0
    and candidate["node_count"] > 1
]

In [34]:
len(actionable_separable_regions)

4

In [37]:
actionable_separable_regions[3]

{'boundary_step': 25192423,
 'boundary_time': 5003.307931452499,
 'boundary_selectors': ('step:25192423',),
 'boundary_selector_kinds': ('step',),
 'region_key': ((242703646.0, 242704061.0),),
 'intervals': ((242703646.0, 242704061.0),),
 'left': 242703646.0,
 'right': 242704061.0,
 'material_length': 415.0,
 'span': 415.0,
 'contiguous': True,
 'lower_frontier_anchor_node_ids': (45956760, 45997789, 47388228),
 'original_suffix_event_indices': (25199373,),
 'mutable_parent_node_ids': (2745970,),
 'original_terminal_lineage_ids': (2745970,),
 'lineage_count': 3,
 'event_count': 1,
 'node_count': 4,
 'terminal_lineage_count': 1,
 'closure_verified': True,
 'eligible': True,
 'rejection_reasons': (),
 'candidate_id': 'step-25192423-region-007354'}

## Live Objects

- `synthetic_result` and `synthetic_arg` contain the synthetic full ARG.
- `trace` contains the complete reversible event history.
- `terminal_state` remains at the complete ARG.
- `traceback_state` can be moved repeatedly with `move_traceback_to_step`, `move_traceback_to_time`, and `return_traceback_to_terminal`.
- Call `traceback_state.compact_active_frontier()` only when exact lineage intervals need to be materialized.
- `separable_boundary_results` contains the complete per-boundary catalogs and scan status.
- `separable_region_candidates` contains every strict closure, including noncontiguous and whole-sequence closures.
- `eligible_separable_regions` contains only closure-verified, contiguous, proper subregions. Candidates at one boundary form an independent partition; different boundaries are alternative catalogs.